In [ ]:
from tensorflow.keras.layers import Input, Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np

# === CONFIG ===
image_size = (32, 32)
batch_size = 32
num_classes = 2  # Based on your previous code

# === LOAD DATA ===
# (This section is identical to your previous code)
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Assuming your paths are still 'Project/Image/train' and 'Project/Image/test'
# Make sure your paths are correct for your Windows environment
train_generator = train_datagen.flow_from_directory(
    r"Project\Image\train",  # Using a raw string (r"...") is safer for Windows paths
    target_size=image_size,
    batch_size=batch_size,
    class_mode='sparse',
    color_mode='rgb' # Explicitly state 3-channel color
)

test_generator = test_datagen.flow_from_directory(
    r"Project\Image\test",
    target_size=image_size,
    batch_size=batch_size,
    class_mode='sparse',
    color_mode='rgb'
)

# === BUILD VGG-STYLE MODEL ===
# This is the new model architecture inspired by VGG16
# VGG Principle: Stack 3x3 convs, then pool. Increase filters as size decreases.

input_shape = (image_size[0], image_size[1], 3)
inputs = Input(shape=input_shape)

# --- Block 1 ---
# Two 3x3 convs with 32 filters, followed by pooling
# Input: 32x32x3
x = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D(pool_size=(2, 2))(x)
# Output: 16x16x32

# --- Block 2 ---
# Two 3x3 convs with 64 filters, followed by pooling
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D(pool_size=(2, 2))(x)
# Output: 8x8x64

# --- Block 3 ---
# Two 3x3 convs with 128 filters, followed by pooling
x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D(pool_size=(2, 2))(x)
# Output: 4x4x128

# --- Classifier Head ---
# Flatten the 3D feature map into a 1D vector
x = Flatten()(x)
# A large dense layer (like in VGG)
x = Dense(512, activation='relu')(x)
# Dropout is added to prevent overfitting, which is common in large models
x = Dropout(0.5)(x)
# Output layer with softmax for classification
outputs = Dense(num_classes, activation='softmax')(x)

# --- Assemble the Model ---
model = Model(inputs, outputs)

# === COMPILE, TRAIN, AND EVALUATE ===
# (This section is identical to your previous code)

model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

model.summary()

# Train the model
history = model.fit(train_generator, 
                    epochs=10,
                    validation_data=test_generator)

# Evaluate the model
loss, accuracy = model.evaluate(test_generator)
print(f"Final Test Loss: {loss}")
print(f"Final Test Accuracy: {accuracy}")

# === VISUALIZE PREDICTIONS ===
# (This section is identical to your previous code)

# Get a batch of test images
try:
    test_images, test_labels = next(test_generator)
    
    # Check if we got enough images for a 5x5 grid
    num_images_to_plot = min(25, len(test_images))
    if num_images_to_plot == 0:
        print("Test generator is empty or could not provide images.")
    else:
        # Get predictions
        pred_probs = model.predict(test_images)
        pred_labels = np.argmax(pred_probs, axis=1)

        # Get class names
        class_names = list(test_generator.class_indices.keys())

        # Plot images
        plt.figure(figsize=(12, 12))
        for i in range(num_images_to_plot):
            plt.subplot(5, 5, i + 1)
            
            # Clip image values to [0, 1] range for correct display
            img = np.clip(test_images[i], 0, 1) 
            
            true_label = int(test_labels[i])
            pred_label = pred_labels[i]
            
            plt.imshow(img)
            plt.title(f"Pred: {class_names[pred_label]}\nTrue: {class_names[true_label]}")
            plt.axis("off")

        plt.tight_layout()
        plt.show()

except StopIteration:
    print("Could not get a batch of images from the test generator. Is the test directory empty?")
